# PagedAttention and the KV Cache, Plainly

Companion notebook for **"PagedAttention and the KV Cache, Plainly: Why Memory, Not FLOPs, Caps Your Concurrency"** (GDE series #2, Nov 10, 2026).

The core claim: when you serve an LLM, the thing that runs out first is almost never compute. It is KV-cache memory. This notebook does the memory arithmetic for three real model sizes, reproduces the fragmentation that PagedAttention (SOSP 2023, Kwon et al.) removes, prices the Nov 5, 2026 block-size change (16 → 64 tokens, vLLM PR #63231), and turns the result into a concurrency budget for an H100 and an H200.

Runs anywhere — no GPU, no vLLM install needed. The last cell prints the exact `vllm serve` commands to run on your GPU box, pinned to vLLM 0.33.0.

## 1. KV cache per token — the one formula

For a standard transformer, every token costs

    bytes/token = 2 (K and V) x layers x KV heads x head dim x bytes per value

Grouped-query attention (GQA) is why the KV-head count matters: Llama 3.1 70B has 64 query heads but only 8 KV heads, so it stores 8x less than a naive multi-head design. DeepSeek V3 goes further with Multi-head Latent Attention (MLA): one 576-dim compressed latent per layer instead of separate K and V per head.

In [ ]:
GIB = 1024 ** 3

def kv_bytes_per_token(layers, kv_heads, head_dim, dtype_bytes=2):
    return 2 * layers * kv_heads * head_dim * dtype_bytes

# (model, layers, kv_heads, head_dim) — configs from the public model cards
MODELS = [
    ("Llama 3.1 8B",  32, 8, 128),
    ("Qwen3 32B",     64, 8, 128),
    ("Llama 3.1 70B", 80, 8, 128),
]
for name, layers, kv_heads, hd in MODELS:
    bpt = kv_bytes_per_token(layers, kv_heads, hd)
    print(f"{name:14s} {bpt:>7,} B/token = {bpt/1024:5.0f} KB/token"
          f"   | 8K ctx: {bpt*8192/GIB:5.2f} GiB   | 128K ctx: {bpt*131072/GIB:5.1f} GiB")

# DeepSeek V3 MLA: 61 layers x one 576-dim compressed latent, bf16
mla_bpt = 61 * 576 * 2
print(f"{'DeepSeek V3 (MLA)':14s} {mla_bpt:>7,} B/token = {mla_bpt/1024:5.1f} KB/token"
      f"   | 8K ctx: {mla_bpt*8192/GIB:5.2f} GiB   | 128K ctx: {mla_bpt*131072/GIB:5.1f} GiB")

assert kv_bytes_per_token(32, 8, 128) == 131072    # 128 KB
assert kv_bytes_per_token(64, 8, 128) == 262144    # 256 KB
assert kv_bytes_per_token(80, 8, 128) == 327680    # 320 KB
assert mla_bpt == 70272                            # ~68.6 KB
# MLA stores 4.7x less per token than a 70B GQA model at the same layer count
assert abs(327680 / mla_bpt - 4.66) < 0.01
print("\nA 70B model's KV cache at 128K context (40 GiB) is bigger than most")
print("whole 8B models. That is the memory wall, in one number.")

## 2. Fragmentation — why contiguous allocation wastes most of the pool

The naive way to serve: reserve `max_model_len` of KV cache per request, up front, in one contiguous slab. A request that ends at 500 tokens still holds 8,192 tokens' worth of memory. The PagedAttention paper (SOSP 2023) reports 60–80% of KV memory wasted this way, and 2–4x higher throughput once paging removes it.

Simulation below: 200 requests with total lengths uniform in [500, 8000], `max_model_len` 8,192. Naive reserves 8,192 per request; paged allocates 16-token blocks on demand.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
lengths = rng.integers(500, 8001, size=200)
MAX_LEN = 8192

naive_alloc = np.full_like(lengths, MAX_LEN)
paged_alloc = np.ceil(lengths / 16) * 16
naive_waste = (naive_alloc - lengths) / naive_alloc
paged_waste = (paged_alloc - lengths) / paged_alloc
print(f"requests: 200, lengths 500-8,000 tokens (mean {lengths.mean():.0f})")
print(f"naive waste:  mean {naive_waste.mean():.1%}, total reserved {naive_alloc.sum()/1e6:.2f}M tokens")
print(f"paged waste:  mean {paged_waste.mean():.2%}, total reserved {paged_alloc.sum()/1e6:.2f}M tokens")
print(f"paging reserves {naive_alloc.sum()/paged_alloc.sum():.2f}x less memory for the same requests")
assert 0.40 < naive_waste.mean() < 0.60   # 500-8000 uniform vs 8192 cap
assert paged_waste.mean() < 0.01          # only the ragged last block
assert paged_alloc.sum() < naive_alloc.sum()

# the paper's headline: waste < 4% with paging (ours: 0.26%), throughput 2-4x
print("\nPaper (SOSP 2023, Kwon et al.): waste 60-80% -> <4%, throughput 2-4x at the same latency.")
print("Our sim lands in the same band; the exact naive % depends on the length mix.")

## 3. The block-size change — 16 → 64 tokens (PR #63231, Nov 5, 2026)

vLLM's default block size was 16 tokens since the PagedAttention paper. PR #63231 (merged Nov 5, 2026) moves it to 64 to cut per-block bookkeeping. The trade: a request's last block is ragged, so the tail waste grows. For a random-length request the expected tail waste is (B−1)/2 tokens — 7.5 at B=16, 31.5 at B=64. A maintainer estimated ~24K fewer tokens of KV capacity at concurrency 1,024. The cell below derives that number: 24 extra wasted tokens per request × 1,024 requests = 24,576.

Two operational consequences land with it: prefix-cache hits now fall on 64-token boundaries (a 40-token shared prompt no longer produces a reusable block), and KV-connector `block_size` values that aren't multiples of 64 fail validation at startup. ROCm keeps 16, to stay on its native kernel.

In [ ]:
def expected_tail_waste(block_size):
    """Mean ragged-tail waste for a random-length request: (B-1)/2 tokens."""
    return (block_size - 1) / 2

for b in (16, 64):
    print(f"block size {b:2d}: avg tail waste {expected_tail_waste(b):5.1f} tokens,"
          f" worst case {b-1} tokens")
extra = expected_tail_waste(64) - expected_tail_waste(16)
print(f"\nextra waste per request at 64 vs 16: {extra:.0f} tokens (average)")
for conc in (256, 512, 1024, 2048):
    print(f"  at concurrency {conc:5d}: {extra * conc:,.0f} tokens of capacity lost")
assert extra == 24.0
assert extra * 1024 == 24576   # ~24K — matches the maintainer's estimate
print("\n24K tokens sounds small until it is 24K tokens of your 70B model's")
print("320 KB/token cache — about 7.5 GiB of HBM at concurrency 1,024.")

## 4. Concurrency — the KV pool is what's left after the weights

vLLM pre-allocates `gpu_memory_utilization` (default 0.9; 0.92 in the latest docs, 1.0 with the new extensible KV cache) of the GPU. Weights load first; everything left is the KV block pool. Divide by the per-request cost at your context length and you get your concurrency ceiling — before a single request arrives.

The dramatic case: Llama 3.1 70B in FP8 weighs 70 GB. On an 80 GB H100, 0.9 × 80 − 70 = 2 GB of KV pool — not enough for even one 8K-token conversation (2.5 GB). The same model on a 141 GB H200 fits 22. The 8B model on the same H100 fits 56. The model picks your concurrency; the GPU just sets the ceiling.

In [ ]:
def concurrency(gpu_gib, weights_gib, bpt, ctx=8192, util=0.9):
    pool = util * gpu_gib - weights_gib
    per_req = ctx * bpt / GIB
    return pool, per_req, int(pool // per_req) if pool > 0 else 0

# (label, gpu GiB, weights GiB, bytes/token)
CASES = [
    ("H100 80GB  + Llama 3.1 8B (bf16)",   80, 16, 131072),
    ("H100 80GB  + Llama 3.1 70B (FP8)",   80, 70, 327680),
    ("H200 141GB + Llama 3.1 8B (bf16)",  141, 16, 131072),
    ("H200 141GB + Llama 3.1 70B (FP8)",  141, 70, 327680),
    ("B200 192GB + Llama 3.1 70B (bf16)", 192, 140, 327680),
]
for label, gpu, w, bpt in CASES:
    pool, per, n = concurrency(gpu, w, bpt)
    print(f"{label}: KV pool {pool:6.1f} GiB, {per:.2f} GiB/req -> {n:3d} concurrent 8K requests")
results = [concurrency(g, w, b)[2] for _, g, w, b in CASES]
assert results == [56, 0, 110, 22, 13]
print("\nZero is not a bug: the 70B FP8 model cannot serve even one 8K conversation")
print("on an 80 GB H100 at util 0.9. You need a bigger GPU, a smaller model,")
print("FP8 weights plus a shorter context, or tensor parallelism across 2 GPUs.")

## 5. Prefix caching — the system prompt, stored once

PagedAttention's blocks are content-addressed: identical prefixes hash to the same blocks, so a shared system prompt is stored once and referenced by every request. `enable_prefix_caching` defaults to True in current vLLM. The saving scales with (N−1) × prompt tokens — for a RAG system with a 2K-token system prompt and 100 concurrent requests on a 70B model, that is ~60 GiB of KV you do not allocate twice.

Caveat from section 3: with the new 64-token default, a shared prefix only produces reusable blocks at 64-token boundaries — a 2,000-token prompt yields 31 full 64-token blocks that are reusable, with only the final 16-token tail unshared; short shared prefixes under 64 tokens may not cache at all.

In [ ]:
def prefix_cache_saving(prompt_tokens, n_requests, bpt):
    """GiB of KV saved by storing one shared prefix once instead of N times."""
    return (n_requests - 1) * prompt_tokens * bpt / GIB

for prompt, n in ((2000, 100), (800, 100), (2000, 1000)):
    saved = prefix_cache_saving(prompt, n, 327680)
    print(f"70B, {prompt}-token shared prefix x {n} requests: saves {saved:.1f} GiB of KV")
assert abs(prefix_cache_saving(2000, 100, 327680) - 60.4) < 0.1

# when the pool runs dry anyway: vLLM preempts. Recompute drops the KV and
# redoes the prefill later; swap offloads blocks to CPU RAM. Both cost latency,
# neither changes the answer — but a preempted request's TTFT explodes.
print("\nIf the pool still runs dry, vLLM preempts the lowest-priority request:")
print("  recompute = drop its KV, redo prefill on reschedule (default)")
print("  swap      = offload blocks to CPU RAM (--swap-space, default 4 GiB)")
print("Preemption is the symptom. The cure is the arithmetic in section 4.")

## 6. Run it yourself — the `vllm serve` commands

The notebook's arithmetic needs no GPU. The commands below do — run them on your GPU box with vLLM 0.33.0 (pinned, per the GDE #1 post). `gpu-memory-utilization` is the fraction in section 4; `max-model-len` caps the context window and therefore the size of the KV block pool; `enable-prefix-caching` is on by default in 0.33. Watch the startup log line `GPU KV cache size:` — it is the pool this notebook computed, measured on your hardware.

In [ ]:
MODEL = "meta-llama/Llama-3.1-8B-Instruct"   # 8B bf16: 16 GiB weights, 128 KB/token
cmds = f'''# vLLM 0.33.0 — PagedAttention is on by default; these flags size the pool
pip install "vllm==0.33.0"

# H100 80GB: 0.9 x 80 = 72 GiB budget, 16 GiB weights -> ~56 GiB KV pool
vllm serve {MODEL} --port 8000 \
    --gpu-memory-utilization 0.90 \
    --max-model-len 8192 \
    --enable-prefix-caching

# check the pool vLLM actually measured (compare with section 4: 56 GiB)
#   "GPU KV cache size: 56.00 GiB" in the startup log

# two identical system prompts -> the second request reuses the first's blocks
curl http://localhost:8000/v1/completions -H "Content-Type: application/json" -d '{{"model": "{MODEL}", "prompt": "You are a helpful assistant.\n\nWhat is 2+2?", "max_tokens": 8}}'
'''
print(cmds)
assert "vllm==0.33.0" in cmds and "vllm serve" in cmds
assert "--gpu-memory-utilization 0.90" in cmds and "--max-model-len 8192" in cmds
print("Expected: a 200, a short completion, and a startup log line reading")
print("'GPU KV cache size' near 56 GiB — the pool this notebook predicted.")